In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # adjust if needed
sys.path.insert(0, str(PROJECT_ROOT))

In [5]:
from utils.visualization import DisplayUSD, DisplayCode
from utils.helperfunctions import create_new_stage

In [30]:
from pxr import Usd, UsdGeom, UsdShade, Sdf, Gf

# ------------------------------------------------------------
# 1. Create stage
# ------------------------------------------------------------
stage = Usd.Stage.CreateNew("table_scene3.usda")
UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.z)
UsdGeom.SetStageMetersPerUnit(stage, 1.0)

# ------------------------------------------------------------
# 2. Helper: simple colored material
# ------------------------------------------------------------
def create_color_material(stage, name, color):
    mat = UsdShade.Material.Define(stage, f"/Materials/{name}")

    shader = UsdShade.Shader.Define(
        stage, f"/Materials/{name}/PBR"
    )
    shader.CreateIdAttr("UsdPreviewSurface")
    shader.CreateInput(
        "diffuseColor", Sdf.ValueTypeNames.Color3f
    ).Set(color)
    shader.CreateInput(
        "roughness", Sdf.ValueTypeNames.Float
    ).Set(0.4)
    shader.CreateInput(
        "metallic", Sdf.ValueTypeNames.Float
    ).Set(0.0)

    mat.CreateSurfaceOutput().ConnectToSource(
        shader.ConnectableAPI(), "surface"
    )
    return mat

# ------------------------------------------------------------
# 3. Materials (vivid, readable)
# ------------------------------------------------------------
wood_mat   = create_color_material(stage, "Wood",   Gf.Vec3f(0.55, 0.27, 0.07))
red_mat    = create_color_material(stage, "Red",    Gf.Vec3f(0.9, 0.1, 0.1))
green_mat  = create_color_material(stage, "Green",  Gf.Vec3f(0.1, 0.8, 0.2))
blue_mat   = create_color_material(stage, "Blue",   Gf.Vec3f(0.1, 0.3, 0.9))
yellow_mat = create_color_material(stage, "Yellow", Gf.Vec3f(0.95, 0.85, 0.1))

# ------------------------------------------------------------
# 4. Helper: cube with correct transform order
# ------------------------------------------------------------
def add_cube(path, scale, translate, material):
    geom = UsdGeom.Cube.Define(stage, path)
    geom.CreateSizeAttr(1.0)
    geom.AddScaleOp().Set(scale)        # scale
    geom.AddTranslateOp().Set(translate)  # then translate
    UsdShade.MaterialBindingAPI(geom).Bind(material)
    return geom

# ------------------------------------------------------------
# 5. Table (top + legs)
# ------------------------------------------------------------
table_height = 0.75
table_top_thickness = 0.1
table_top_z = table_height
table_top_top_z = table_top_z + table_top_thickness / 2
table_top_bottom_z = table_top_z - table_top_thickness / 2

# Table top
add_cube(
    "/Table/Top",
    Gf.Vec3f(2.0, 1.2, table_top_thickness),
    Gf.Vec3f(0, 0, table_top_z),
    wood_mat
)

# Table legs
leg_height = table_top_bottom_z
leg_center_z = leg_height / 2

for i, (x, y) in enumerate(leg_positions):
    add_cube(
        f"/Table/Leg_{i}",
        Gf.Vec3f(0.1, 0.1, leg_height),
        Gf.Vec3f(x, y, leg_center_z),
        wood_mat
    )
# ------------------------------------------------------------
# 6. Boxes on top of table
# ------------------------------------------------------------
def add_box(path, size, x, y, material):
    box_height = size[2]  # Z dimension
    z = table_top_top_z + box_height / 2
    add_cube(
        path,
        size,
        Gf.Vec3f(x, y, z),
        material
    )

add_box("/Boxes/RedBox",    Gf.Vec3f(0.25, 0.25, 0.25), -0.5,  0.0, red_mat)
add_box("/Boxes/GreenBox", Gf.Vec3f(0.20, 0.30, 0.20),  0.0, -0.3, green_mat)
add_box("/Boxes/BlueBox",  Gf.Vec3f(0.30, 0.20, 0.30),  0.5,  0.3, blue_mat)
add_box("/Boxes/YellowBox",Gf.Vec3f(0.20, 0.20, 0.20),  0.0,  0.4, yellow_mat)
# ------------------------------------------------------------
# 7. Ground plane (visual grounding)
# ------------------------------------------------------------
add_cube(
    "/Ground",
    Gf.Vec3f(10, 10, 0.05),
    Gf.Vec3f(0, 0, -0.025),
    create_color_material(stage, "GroundGray", Gf.Vec3f(0.7, 0.7, 0.7))
)

# ------------------------------------------------------------
# 8. Save
# ------------------------------------------------------------
stage.GetRootLayer().Save()
print("Table scene with vivid boxes created.")


Table scene with vivid boxes created.


In [31]:
from pxr import Usd, UsdGeom, Gf

file_path = "attributes_ex2.usda"
stage: Usd.Stage = Usd.Stage.CreateNew(file_path)

world_xform: UsdGeom.Xform = UsdGeom.Xform.Define(stage, "/World")

sphere: UsdGeom.Sphere = UsdGeom.Sphere.Define(stage, world_xform.GetPath().AppendPath("Sphere"))
cube: UsdGeom.Cube = UsdGeom.Cube.Define(stage, world_xform.GetPath().AppendPath("Cube"))
UsdGeom.XformCommonAPI(cube).SetTranslate(Gf.Vec3d(5, 0, 0))

# Get the attributes of the cube prim
cube_attrs = cube.GetPrim().GetAttributes()
for attr in cube_attrs:
    print(attr)

# Get the size, display color, and extent attributes of the cube
cube_size: Usd.Attribute = cube.GetSizeAttr()
cube_displaycolor: Usd.Attribute = cube.GetDisplayColorAttr()
cube_extent: Usd.Attribute = cube.GetExtentAttr()

print(f"Size: {cube_size.Get()}")
print(f"Display Color: {cube_displaycolor.Get()}")
print(f"Extent: {cube_extent.Get()}")

stage.Save()

Usd.Prim(</World/Cube>).GetAttribute('doubleSided')
Usd.Prim(</World/Cube>).GetAttribute('extent')
Usd.Prim(</World/Cube>).GetAttribute('orientation')
Usd.Prim(</World/Cube>).GetAttribute('primvars:displayColor')
Usd.Prim(</World/Cube>).GetAttribute('primvars:displayOpacity')
Usd.Prim(</World/Cube>).GetAttribute('purpose')
Usd.Prim(</World/Cube>).GetAttribute('size')
Usd.Prim(</World/Cube>).GetAttribute('visibility')
Usd.Prim(</World/Cube>).GetAttribute('xformOp:translate')
Usd.Prim(</World/Cube>).GetAttribute('xformOpOrder')
Size: 2.0
Display Color: None
Extent: [(-1, -1, -1), (1, 1, 1)]


In [32]:
DisplayUSD("attributes_ex2.usda")

In [6]:
from pxr import Usd, UsdGeom, UsdPhysics, Gf

def build_stage(stage: Usd.Stage):
    # Stage metadata
    # stage.SetUpAxis(UsdGeom.Tokens.z)
    # stage.SetTimeCodesPerSecond(24)

    # Root prim
    UsdGeom.Xform.Define(stage, "/World")

    size = 25.0

    # --------------------------------------------------
    # Circular Kinematic Conveyor (Cylinder)
    # --------------------------------------------------
    cylinder_path = "/World/CircularConveyor"
    cylinder = UsdGeom.Cylinder.Define(stage, cylinder_path)
    cylinder.CreateRadiusAttr(size * 8.0)
    cylinder.CreateHeightAttr(size * 0.5)
    cylinder.GetAxisAttr().Set(UsdGeom.Tokens.z)

    cylinder_prim = stage.GetPrimAtPath(cylinder_path)

    # Physics schemas
    UsdPhysics.CollisionAPI.Apply(cylinder_prim)
    rb = UsdPhysics.RigidBodyAPI.Apply(cylinder_prim)
    rb.CreateKinematicEnabledAttr(True)
    rb.CreateAngularVelocityAttr(Gf.Vec3f(0.0, 0.0, size * 0.25))

    # --------------------------------------------------
    # Linear Kinematic Conveyor (Box)
    # --------------------------------------------------
    linear_path = "/World/LinearConveyor"
    linear = UsdGeom.Cube.Define(stage, linear_path)

    xf = UsdGeom.Xformable(linear)
    xf.AddTranslateOp().Set(Gf.Vec3f(0, 0, size))
    xf.AddScaleOp().Set(Gf.Vec3f(0.5 * size, 10.0 * size, 0.5 * size))

    linear_prim = stage.GetPrimAtPath(linear_path)

    UsdPhysics.CollisionAPI.Apply(linear_prim)
    rb_linear = UsdPhysics.RigidBodyAPI.Apply(linear_prim)
    rb_linear.CreateKinematicEnabledAttr(True)

    velocity_attr = rb_linear.CreateVelocityAttr()
    v = Gf.Vec3f(0.0, size * 3.0, 0.0)

    velocity_attr.Set(v, time=0)
    velocity_attr.Set(v, time=30)
    velocity_attr.Set(-v, time=31)
    velocity_attr.Set(-v, time=60)

    # --------------------------------------------------
    # Dynamic Box
    # --------------------------------------------------
    box_path = "/World/DynamicBox"
    box = UsdGeom.Cube.Define(stage, box_path)

    xf_box = UsdGeom.Xformable(box)
    xf_box.AddTranslateOp().Set(Gf.Vec3f(0, 0, size * 2.0))
    xf_box.AddScaleOp().Set(Gf.Vec3f(size * 0.5))

    box_prim = stage.GetPrimAtPath(box_path)
    UsdPhysics.CollisionAPI.Apply(box_prim)
    UsdPhysics.RigidBodyAPI.Apply(box_prim)


if __name__ == "__main__":
    stage = Usd.Stage.CreateNew("./getting_started/conveyor_usd_only.usda")
    build_stage(stage)
    stage.GetRootLayer().Save()


In [7]:
DisplayUSD("./getting_started/conveyor_usd_only.usda",show_usd_code=True)

USD file "./getting_started/conveyor_usd_only.usda" not found.


In [26]:
from pxr import (
    Usd, UsdGeom, UsdShade, UsdUtils,
    Sdf, Gf
)

# ------------------------------------------------------------
# 1. Create stage
# ------------------------------------------------------------
stage = Usd.Stage.CreateNew("conveyor3.usda")
UsdGeom.SetStageUpAxis(stage, UsdGeom.Tokens.z)
UsdGeom.SetStageMetersPerUnit(stage, 1.0)

# ------------------------------------------------------------
# 2. Conveyor geometry
# ------------------------------------------------------------
root = UsdGeom.Xform.Define(stage, "/Conveyor")

def add_belt(path, scale, translate, rotate):
    geom = UsdGeom.Cube.Define(stage, path)
    geom.CreateSizeAttr(1.0)
    geom.AddScaleOp().Set(scale)
    geom.AddTranslateOp().Set(translate)
    geom.AddRotateXYZOp().Set(rotate)
    return geom

belt1 = add_belt(
    "/Conveyor/Straight",
    Gf.Vec3f(6, 1.2, 0.2),
    Gf.Vec3f(0, 0, 1),
    Gf.Vec3f(0, 0, 0)
)

belt2 = add_belt(
    "/Conveyor/Slope",
    Gf.Vec3f(5, 1.2, 0.2),
    Gf.Vec3f(6, 0, 2.3),
    Gf.Vec3f(0, -20, 0)
)

belt3 = add_belt(
    "/Conveyor/Turn",
    Gf.Vec3f(4, 1.2, 0.2),
    Gf.Vec3f(10, 2, 2.8),
    Gf.Vec3f(0, 0, 90)
)

# ------------------------------------------------------------
# 3. Add UVs (MANDATORY for texturing)
# ------------------------------------------------------------
def add_uvs(geom):
    prim = geom.GetPrim()
    pv = UsdGeom.PrimvarsAPI(prim)

    st = pv.CreatePrimvar(
        "st",
        Sdf.ValueTypeNames.TexCoord2fArray,
        UsdGeom.Tokens.vertex
    )

    st.Set([
        Gf.Vec2f(0, 0),
        Gf.Vec2f(1, 0),
        Gf.Vec2f(1, 1),
        Gf.Vec2f(0, 1),
        Gf.Vec2f(0, 0),
        Gf.Vec2f(1, 0),
        Gf.Vec2f(1, 1),
        Gf.Vec2f(0, 1),
    ])

for g in (belt1, belt2, belt3):
    add_uvs(g)

# ------------------------------------------------------------
# 4. Material + PBR shader (OpenUSD canonical)
# ------------------------------------------------------------
material = UsdShade.Material.Define(stage, "/Materials/ConveyorMat")

pbrShader = UsdShade.Shader.Define(
    stage, "/Materials/ConveyorMat/PBRShader"
)
pbrShader.CreateIdAttr("UsdPreviewSurface")
pbrShader.CreateInput("roughness", Sdf.ValueTypeNames.Float).Set(0.45)
pbrShader.CreateInput("metallic", Sdf.ValueTypeNames.Float).Set(0.05)

material.CreateSurfaceOutput().ConnectToSource(
    pbrShader.ConnectableAPI(), "surface"
)

# ------------------------------------------------------------
# 5. Primvar reader (st)
# ------------------------------------------------------------
stReader = UsdShade.Shader.Define(
    stage, "/Materials/ConveyorMat/stReader"
)
stReader.CreateIdAttr("UsdPrimvarReader_float2")
stReader.CreateInput(
    "varname", Sdf.ValueTypeNames.Token
).Set("st")

# ------------------------------------------------------------
# 6. Diffuse texture sampler
# ------------------------------------------------------------
diffuseTextureSampler = UsdShade.Shader.Define(
    stage, "/Materials/ConveyorMat/diffuseTexture"
)
diffuseTextureSampler.CreateIdAttr("UsdUVTexture")

diffuseTextureSampler.CreateInput(
    "file", Sdf.ValueTypeNames.Asset
).Set("USDLogoLrg.png")  # place next to this USD

diffuseTextureSampler.CreateInput(
    "st", Sdf.ValueTypeNames.Float2
).ConnectToSource(
    stReader.ConnectableAPI(), "result"
)

diffuseTextureSampler.CreateInput(
    "sourceColorSpace", Sdf.ValueTypeNames.Token
).Set("sRGB")

diffuseTextureSampler.CreateInput(
    "fallback", Sdf.ValueTypeNames.Float3
).Set(Gf.Vec3f(0.6, 0.6, 0.6))

diffuseTextureSampler.CreateOutput(
    "rgb", Sdf.ValueTypeNames.Float3
)

# ------------------------------------------------------------
# 7. Connect texture → PBR
# ------------------------------------------------------------
pbrShader.CreateInput(
    "diffuseColor", Sdf.ValueTypeNames.Color3f
).ConnectToSource(
    diffuseTextureSampler.ConnectableAPI(), "rgb"
)

# ------------------------------------------------------------
# 8. Bind material
# ------------------------------------------------------------
for g in (belt1, belt2, belt3):
    UsdShade.MaterialBindingAPI(g).Bind(material)

# ------------------------------------------------------------
# 9. Save
# ------------------------------------------------------------
stage.GetRootLayer().Save()
print("USD conveyor with textured PBR material created.")

USD conveyor with textured PBR material created.
